In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pathlib import Path
# read in all the data for yellow taxis
spark = SparkSession.builder \
.appName("Read Parquet") \
    .config("spark.driver.memory", "12g")\
        .getOrCreate()
YELLOW_DIR = Path('data/year-2026-yellow/')
# read in all the data across the times
from pathlib import Path

data_root = Path("data")
pattern = "year-*-yellow"  # matches year-2026-yellow, year-2025-yellow, etc.
dirs = sorted([d for d in data_root.glob(pattern) if d.is_dir()])

parquet_files = []
for d in dirs:
    parquet_files.extend(sorted([p for p in d.glob("*.parquet") if p.is_file()]))

if not parquet_files:
    raise FileNotFoundError(f"No parquet files found under {data_root}/{pattern}")

yellow_data = None
for p in parquet_files:
    df = spark.read.parquet(str(p))
    if yellow_data is None:
        yellow_data = df
    else:
        # use unionByName to be robust to column ordering / missing columns
        yellow_data = yellow_data.unionByName(df, allowMissingColumns=True)

# yellow_data is the big row-wise union of all parquet files
print(f"Combined {len(parquet_files)} parquet files into dataframe with {yellow_data.count()} rows")
yellow_data.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/23 15:03:28 WARN Utils: Your hostname, aydin-khan-desktop, resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlp11s0)
26/08/23 15:03:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/23 15:03:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where app

Combined 41 parquet files into dataframe with 147201830 rows
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



# Cleaning the dataset

In [2]:
# get rid of rows where passenger count is 0 or null
yellow_data = yellow_data.filter((F.col("passenger_count") > 0) & (F.col("passenger_count").isNotNull()))
# get rid of rows where trip distance is 0, null or greater than 25
yellow_data = yellow_data.filter((F.col("trip_distance") > 0) & (F.col("trip_distance").isNotNull()) & (F.col("trip_distance") <= 25))
# get rid of rows where the trip duration is 0, null, less than 60 seconds or greater than 10000 seconds
yellow_data = yellow_data.filter((F.col("tpep_dropoff_datetime").isNotNull()) & (F.col("tpep_pickup_datetime").isNotNull()))
yellow_data = yellow_data.withColumn("trip_duration", F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))
yellow_data = yellow_data.filter((F.col("trip_duration") > 60) & (F.col("trip_duration").isNotNull()) & (F.col("trip_duration") < 10000))
yellow_data = yellow_data.filter(~F.col("PULocationID").isin([264, 265, 103, 104, 110]))
# get rid of rows where payment_type is null or not in the array [5, 6]
yellow_data = yellow_data.filter((F.col("payment_type").isNotNull()) & (~F.col("payment_type").isin([5, 6])))
# remove rows where the year in tpep_pickup_datetime is not in the range 2023-2026
yellow_data = yellow_data.withColumn("pickup_year", F.year("tpep_pickup_datetime"))
yellow_data = yellow_data.filter((F.col("pickup_year") >= 2023) & (F.col("pickup_year") <= 2026))
print(yellow_data.count())

120193181


In [ ]:
# save the cleaned data to a new parquet file
yellow_data.write.parquet("cleaned_data/yellow_data.parquet", mode="overwrite")

26/08/23 15:04:10 WARN DAGScheduler: Broadcasting large task binary with size 1229.8 KiB


Next steps
- Get FHV data
- Show FHV demand increasing as yellow taxi demand decreases
- Then justify a course of action: Redirect yellow taxis to the zones where FHVs operate less frequently.
- Get data on employment density in the various zones
- Analyse the effect of employment density on taxi demand, controlling for subway station availability.

In [5]:
pattern = "year-*-fhvhv"  # matches year-2026-yellow, year-2025-yellow, etc.
dirs = sorted([d for d in data_root.glob(pattern) if d.is_dir()])

parquet_files = []
for d in dirs:
    parquet_files.extend(sorted([p for p in d.glob("*.parquet") if p.is_file()]))

if not parquet_files:
    raise FileNotFoundError(f"No parquet files found under {data_root}/{pattern}")

fhvhv_data = None
for p in parquet_files:
    df = spark.read.parquet(str(p))
    if fhvhv_data is None:
        fhvhv_data = df
    else:
        # use unionByName to be robust to column ordering / missing columns
        fhvhv_data = fhvhv_data.unionByName(df, allowMissingColumns=True)

# fhvhv_data is the big row-wise union of all parquet files
print(f"Combined {len(parquet_files)} parquet files into dataframe with {fhvhv_data.count()} rows")
fhvhv_data.printSchema()

Combined 41 parquet files into dataframe with 821546266 rows
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- sha

In [6]:
# data cleaning
# remove rows where driver_pay is null, less than or equal to 0
fhvhv_data = fhvhv_data.filter(fhvhv_data.driver_pay.isNotNull() & (fhvhv_data.driver_pay > 0))
# remove rows where trip time is null or less than 10 seconds
fhvhv_data = fhvhv_data.filter(fhvhv_data.trip_time.isNotNull() & (fhvhv_data.trip_time >= 10))
# remove rows where trip miles is null or 0
fhvhv_data = fhvhv_data.filter(fhvhv_data.trip_miles.isNotNull() & (fhvhv_data.trip_miles > 0))
# remove rows where PULocationID is in the list of invalid IDs
fhvhv_data = fhvhv_data.filter(~F.col("PULocationID").isin([264, 265, 103, 104, 110]))
# do the same for DOLocationID
fhvhv_data = fhvhv_data.filter(~F.col("DOLocationID").isin([264, 265, 103, 104, 110]))

In [8]:
# save the cleaned data to a new parquet file
fhvhv_data.write.mode("overwrite").parquet("cleaned_data/fhvhv_data.parquet")

26/08/23 15:06:26 WARN DAGScheduler: Broadcasting large task binary with size 1192.4 KiB
